Notebook that reads in the 2018 hct_population_coverage csv and the 2023 hct_population_coverage csv and then joins the 2018 data with a census crosswalk in order to translate the census tracts to their 2020 equivalents. Then compares the percentage population between 2018 to 2023 in each census tract in order to test the 2023 output and see if everything looks accurate. 

Read the comparison results out to a csv, the path is in the last cell. Change that to your preferred file path if you want to run this notebook.

In [ ]:
import pandas as pd

df2010 = pd.read_csv(r"t:\60day-TEMP\Brice\hct_population_coverage_2018.csv", dtype={"GEOID": str})
df2020 = pd.read_csv(r"c:\workspace\displacement_index\displacement_index_current\08-Proximity-to-Transit\output\hct_population_coverage_2023.csv", dtype={"GEOID": str})
crosswalk = pd.read_csv(r"c:\workspace\displacement_index\displacement_index_current\nhgis_tr2010_tr2020_53.csv",
                        dtype={"GEOID_2010": str, "GEOID_2020": str})
df2020.columns

Index(['Census2020Tract', 'population', 'population_quarter_mile',
       'percent_pop_quarter_mile'],
      dtype='object')

In [12]:
merged = df2010.merge(
    crosswalk,
    left_on="geoid10",
    right_on="tr2010ge",
    how="left"
)
merged.head()

,geoid10,tractce10,population,population_quarter_mile,percent_pop_quarter_mile,tr2010gj,tr2010ge,tr2020gj,tr2020ge,parea,wt_pop,wt_adult,wt_fam,wt_hh,wt_hu,wt_ownhu,wt_renthu
0,53033022006,22006,3862.0,0.0,0.0,G5300330022006,53033022006,G5300330022006,53033022006,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,53033032320,32320,4488.0,0.0,0.0,G5300330032320,53033032320,G5300330032307,53033032307,0.009535,0.004756,0.004195,0.004153,0.003506,0.003392,0.003084,0.007752
2,53033032320,32320,4488.0,0.0,0.0,G5300330032320,53033032320,G5300330032319,53033032319,0.000174,0.000400,0.000399,0.000496,0.000460,0.000445,0.000473,0.000321
3,53033032320,32320,4488.0,0.0,0.0,G5300330032320,53033032320,G5300330032320,53033032320,0.985818,0.993241,0.993588,0.993361,0.994170,0.994342,0.994433,0.991518
4,53033032320,32320,4488.0,0.0,0.0,G5300330032320,53033032320,G5300330032321,53033032321,0.004473,0.001603,0.001818,0.001990,0.001865,0.001822,0.002009,0.000408


In [15]:
merged["pop2010_weighted"] = merged["population"] * merged["wt_pop"]
merged["pop2010_weighted"]

0       3862.000000
1         21.346017
2          1.797185
3       4457.663501
4          7.193297
           ...     
1488    4093.000000
1489    4378.000000
1490       0.000000
1491       0.000000
1492    4419.000000
Name: pop2010_weighted, Length: 1493, dtype: float64

In [34]:
merged["pop2010_quarter_mile_weighted"] = merged["population_quarter_mile"] * merged["wt_pop"]
merged.columns

Index(['geoid10', 'tractce10', 'population', 'population_quarter_mile',
       'percent_pop_quarter_mile', 'tr2010gj', 'tr2010ge', 'tr2020gj',
       'tr2020ge', 'parea', 'wt_pop', 'wt_adult', 'wt_fam', 'wt_hh', 'wt_hu',
       'wt_ownhu', 'wt_renthu', 'pop2010_weighted',
       'pop2010_quarter_mile_weighted'],
      dtype='object')

In [36]:
df2010_adjusted = (
    merged
    .groupby("tr2020ge", as_index=False)[["pop2010_weighted", "pop2010_quarter_mile_weighted"]]
    .sum()
)
df2010_adjusted.head()

,tr2020ge,pop2010_weighted,pop2010_quarter_mile_weighted
0,53007960100,0.002419,0.0
1,53007960203,2.531784,0.0
2,53033000101,3445.960352,0.0
3,53033000102,4461.039648,0.0
4,53033000201,4119.863981,0.0


In [ ]:
final = df2010_adjusted.merge(
    df2020,
    left_on="tr2020ge",
    right_on="tr2020ge",
    how="inner"
)

In [38]:
final.head()

,tr2020ge,pop2010_weighted,pop2010_quarter_mile_weighted,Census2020Tract,population,population_quarter_mile,percent_pop_quarter_mile
0,53033000101,3445.960352,0.0,5.303300e+10,3670,0.0,0.000000
1,53033000102,4461.039648,0.0,5.303300e+10,4270,0.0,0.000000
2,53033000201,4119.863981,0.0,5.303300e+10,4401,0.0,0.000000
3,53033000202,4240.136019,0.0,5.303300e+10,3989,0.0,0.000000
4,53033000300,2839.000000,774.0,5.303300e+10,2831,848.0,0.299541


In [40]:
final["pop2010_percent_pop_quarter_mile_weighted"] = final["pop2010_quarter_mile_weighted"] / final["pop2010_weighted"]
final.head()

,tr2020ge,pop2010_weighted,pop2010_quarter_mile_weighted,Census2020Tract,population,population_quarter_mile,percent_pop_quarter_mile,pop2010_percent_pop_quarter_mile_weighted
0,53033000101,3445.960352,0.0,5.303300e+10,3670,0.0,0.000000,0.000000
1,53033000102,4461.039648,0.0,5.303300e+10,4270,0.0,0.000000,0.000000
2,53033000201,4119.863981,0.0,5.303300e+10,4401,0.0,0.000000,0.000000
3,53033000202,4240.136019,0.0,5.303300e+10,3989,0.0,0.000000,0.000000
4,53033000300,2839.000000,774.0,5.303300e+10,2831,848.0,0.299541,0.272631


In [ ]:
final["absolute_change"] = final["percent_pop_quarter_mile"] - final["pop2010_percent_pop_quarter_mile_weighted"]

final.to_csv(r"c:\workspace\displacement_index\displacement_index_current\08-Proximity-to-Transit\output\hct_population_coverage_comparison_2010_2020_absolute_change.csv", index=False)